In [1]:
import numpy as np
import scipy.sparse as sps
import scipy.sparse.linalg as spla

class LidDrivenCavitySolver2D:
    """
    Implements a 2D Incompressible Navier-Stokes Solver using the 
    Fractional Step (Projection) Method on a Staggered Grid.
    Designed for structural mapping and transfer to FOWT offshore final projects.
    """
    def __init__(self, Nx, Ny, Lx, Ly, rho, nu, U_lid):
        self.Nx = Nx
        self.Ny = Ny
        self.Lx = Lx
        self.Ly = Ly
        self.rho = rho
        self.nu = nu
        self.U_lid = U_lid
        
        self.dx = Lx / Nx
        self.dy = Ly / Ny
        
        # --- STAGGERED GRID GRIDDING ARCHITECTURE ---
        # u-velocity: resides on East-West cell faces -> (Nx+1, Ny)
        self.u = np.zeros((Nx + 1, Ny))
        self.u_star = np.zeros((Nx + 1, Ny))
        
        # v-velocity: resides on North-South cell faces -> (Nx, Ny+1)
        self.v = np.zeros((Nx, Ny + 1))
        self.v_star = np.zeros((Nx, Ny + 1))
        
        # Pressure: resides at cell centers -> (Nx, Ny)
        self.p = np.zeros((Nx, Ny))
        
        # Build the permanent sparse Pressure-Poisson Operator Matrix
        self.A_pp = self._assemble_pressure_poisson_matrix()

    def _assemble_pressure_poisson_matrix(self):
        """
        PROBLEM 3: Casts the 2D Laplace operator into linear system AP = Q.
        Maps 2D (i, j) coordinates into 1D global vector rows using I = i + j * Nx.
        """
        N_total = self.Nx * self.Ny
        A = sps.lil_matrix((N_total, N_total), dtype=np.float64)
        
        idx_dx = 1.0 / (self.dx**2)
        idx_dy = 1.0 / (self.dy**2)
        
        for j in range(self.Ny):
            for i in range(self.Nx):
                I = i + j * self.Nx
                
                # Enforce boundary modifications for Neumann conditions (Zero Gradient)
                # If an edge face points to the outside wall, the boundary gradient drops out,
                # modifying the center coefficient diagonal weight.
                west_coeff  = idx_dx if i > 0 else 0.0
                east_coeff  = idx_dx if i < self.Nx - 1 else 0.0
                south_coeff = idx_dy if j > 0 else 0.0
                north_coeff = idx_dy if j < self.Ny - 1 else 0.0
                
                # Center diagonal entry
                A[I, I] = -(west_coeff + east_coeff + south_coeff + north_coeff)
                
                # Inject non-zero neighbor coefficients into the sparse grid
                if i > 0:           A[I, I - 1] = west_coeff
                if i < self.Nx - 1: A[I, I + 1] = east_coeff
                if j > 0:           A[I, I - self.Nx] = south_coeff
                if j < self.Ny - 1: A[I, I + self.Nx] = north_coeff
                
        # CRITICAL SHORTCUT: Pin down one arbitrary cell pressure to 0.0 to prevent
        # a singular matrix error caused by pure Neumann boundary conditions.
        A[0, 0] = -1.0 / (self.dx**2)
        
        return A.tocsr()

    def compute_predictor_step(self, dt):
        """
        PROBLEM 1: The Predictor Step. Advances momentum forward in time
        to calculate intermediate velocities (u_star, v_star) without pressure.
        """
        # --- UPDATE U-MOMENTUM FIELD ---
        for j in range(1, self.Ny - 1):
            for i in range(1, self.Nx):
                # Central Differencing for Bending/Diffusion Terms
                u_xx = (self.u[i+1, j] - 2.0*self.u[i, j] + self.u[i-1, j]) / (self.dx**2)
                u_yy = (self.u[i, j+1] - 2.0*self.u[i, j] + self.u[i, j-1]) / (self.dy**2)
                diffusion = self.nu * (u_xx + u_yy)
                
                # Advection terms (interpolating surrounding velocities to face centers)
                u_avg_x = (self.u[i+1, j] + self.u[i, j]) / 2.0
                u_avg_x_left = (self.u[i, j] + self.u[i-1, j]) / 2.0
                adv_x = (u_avg_x**2 - u_avg_x_left**2) / self.dx
                
                v_avg_y = (self.v[i, j+1] + self.v[i-1, j+1]) / 2.0
                v_avg_y_low = (self.v[i, j] + self.v[i-1, j]) / 2.0
                u_avg_y = (self.u[i, j+1] + self.u[i, j]) / 2.0
                u_avg_y_low = (self.u[i, j] + self.u[i, j-1]) / 2.0
                adv_y = (u_avg_y * v_avg_y - u_avg_y_low * v_avg_y_low) / self.dy
                
                self.u_star[i, j] = self.u[i, j] + dt * (diffusion - adv_x - adv_y)

        # --- UPDATE V-MOMENTUM FIELD ---
        for j in range(1, self.Ny):
            for i in range(1, self.Nx - 1):
                v_xx = (self.v[i+1, j] - 2.0*self.v[i, j] + self.v[i-1, j]) / (self.dx**2)
                v_yy = (self.v[i, j+1] - 2.0*self.v[i, j] + self.v[i, j-1]) / (self.dy**2)
                diffusion = self.nu * (v_xx + v_yy)
                
                u_avg_x = (self.u[i+1, j] + self.u[i+1, j-1]) / 2.0
                u_avg_x_left = (self.u[i, j] + self.u[i, j-1]) / 2.0
                v_avg_x = (self.v[i+1, j] + self.v[i, j]) / 2.0
                v_avg_x_left = (self.v[i, j] + self.v[i-1, j]) / 2.0
                adv_x = (u_avg_x * v_avg_x - u_avg_x_left * v_avg_x_left) / self.dx
                
                v_avg_y = (self.v[i, j+1] + self.v[i, j]) / 2.0
                v_avg_y_low = (self.v[i, j] + self.v[i, j-1]) / 2.0
                adv_y = (v_avg_y**2 - v_avg_y_low**2) / self.dy
                
                self.v_star[i, j] = self.v[i, j] + dt * (diffusion - adv_x - adv_y)
                
        # PROBLEM 2: Enforce Wall Boundary conditions on intermediate velocities
        self._apply_velocity_boundary_conditions()

    def solve_pressure_poisson(self, dt):
        """
        PROBLEM 2 & 3: Calculates velocity divergence field as the Poisson RHS 
        source vector (Q) and solves for the correction pressure field.
        """
        N_total = self.Nx * self.Ny
        Q = np.zeros(N_total)
        
        # Build the RHS divergence vector field
        for j in range(self.Ny):
            for i in range(self.Nx):
                I = i + j * self.Nx
                div_u = (self.u_star[i+1, j] - self.u_star[i, j]) / self.dx
                div_v = (self.v_star[i, j+1] - self.v_star[i, j]) / self.dy
                Q[I] = (self.rho / dt) * (div_u + div_v)
                
        # Resolve the sparse linear algebra problem via Direct Umfpack Solvers
        p_vector = spla.spsolve(self.A_pp, Q)
        self.p = p_vector.reshape((self.Nx, self.Ny))

    def compute_corrector_step(self, dt):
        """
        PROBLEM 1: Velocity Correction. Subtracts the pressure gradient
        from the intermediate field to restore perfect mass conservation.
        """
        # Correct horizontal u-velocities
        for j in range(self.Ny):
            for i in range(1, self.Nx):
                p_grad_x = (self.p[i, j] - self.p[i-1, j]) / self.dx
                self.u[i, j] = self.u_star[i, j] - (dt / self.rho) * p_grad_x
                
        # Correct vertical v-velocities        
        for j in range(1, self.Ny):
            for i in range(self.Nx):
                p_grad_y = (self.p[i, j] - self.p[i, j-1]) / self.dy
                self.v[i, j] = self.v_star[i, j] - (dt / self.rho) * p_grad_y
                
        # Enforce hard wall boundary constraints on the corrected arrays
        self._apply_velocity_boundary_conditions()

    def _apply_velocity_boundary_conditions(self):
        """
        Applies zero-penetration and no-slip conditions to all walls.
        The top wall drives the cavity flow via U_lid.
        """
        # Hard No-Slip / Zero-Penetration constraints on physical edges
        self.u[0, :] = 0.0
        self.u[-1, :] = 0.0
        self.v[:, 0] = 0.0
        self.v[:, -1] = 0.0
        
        # Top Wall moving boundary condition (Lid Velocity)
        self.u[:, -1] = self.U_lid

    def step_simulation(self, dt):
        """Executes a single fully coupled Fractional Step Time Iteration."""
        self.compute_predictor_step(dt)
        self.solve_pressure_poisson(dt)
        self.compute_corrector_step(dt)

# --- INVERSION TESTING AND SKELETON EXECUTION ---
if __name__ == "__main__":
    # Define physical conditions matching standard LDC benchmark bounds
    solver = LidDrivenCavitySolver2D(Nx=32, Ny=32, Lx=1.0, Ly=1.0, rho=1.0, nu=0.01, U_lid=1.0)
    
    print("Executing Time Step 1 of the Fractional Step Coupled Navier-Stokes Solver...")
    solver.step_simulation(dt=0.001)
    print("Time step complete! Mass conservation has been implicitly satisfied.")

Executing Time Step 1 of the Fractional Step Coupled Navier-Stokes Solver...
Time step complete! Mass conservation has been implicitly satisfied.
